<a href="https://colab.research.google.com/github/RMPlaysMCYT/Corn-Detection-System/blob/main/python/Notebooks/Corn_Seeds_Detection_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corn Seed Quality Detection System
This page contains the process of the corn seed quality detection system which means it shows on how we produce this datasets

## Imports

In [2]:
import tensorflow as tf
import keras
from keras._tf_keras.keras.preprocessing.image import ImageDataGenerator

In [3]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Cropping Images and Processed ones

In [ ]:
import os
import shutil
import cv2
import numpy as np
from pathlib import Path

base_path = '/content/drive/MyDrive/CornSeeds/NewCornData'
output_path = '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data'

# Create output folders
processed_folders = ['Healthy_Processed', 'Unhealthy_Processed']
for folder in processed_folders:
    os.makedirs(os.path.join(output_path, folder), exist_ok=True)

def zoom_and_resize_image(image_path, target_size=(256, 256), zoom_factor=1.2):
    """
    Zoom into the center of the image and resize to target size
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Get original dimensions
    h, w = img.shape[:2]

    # Calculate crop dimensions (zoom in)
    new_h = int(h / zoom_factor)
    new_w = int(w / zoom_factor)

    # Calculate center crop coordinates
    start_h = (h - new_h) // 2
    start_w = (w - new_w) // 2

    # Crop the center
    cropped = img[start_h:start_h + new_h, start_w:start_w + new_w]

    # Resize to target size
    resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LANCZOS4)

    return resized

def remove_background_and_zoom(image_path, target_size=(256, 256)):
    """
    Advanced: Remove background and focus on the corn seed
    """
    img = cv2.imread(image_path)
    if img is None:
        return None

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Apply threshold to find the seed (adjust threshold as needed)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Find the largest contour (assuming it's the seed)
        largest_contour = max(contours, key=cv2.contourArea)

        # Get bounding box
        x, y, w, h = cv2.boundingRect(largest_contour)

        # Add some padding
        padding = 20
        x = max(0, x - padding)
        y = max(0, y - padding)
        w = min(img.shape[1] - x, w + 2 * padding)
        h = min(img.shape[0] - y, h + 2 * padding)

        # Crop to the seed
        cropped = img[y:y+h, x:x+w]

        # Resize
        resized = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LANCZOS4)
        return resized

    # Fallback: use center zoom if no contour found
    return zoom_and_resize_image(image_path, target_size)

# Process all images
total_processed = 0

# Map source folders to output folders
folder_mapping = {
    'Healthy': {
        'users': ['Ronnel', 'Marjorie'],
        'output': 'Healthy_Processed'
    },
    'Unhealthy': {
        'users': ['Rodriguez', 'Ronnel', 'Marjorie'],
        'output': 'Unhealthy_Processed'
    }
}

for source_category, config in folder_mapping.items():
    for user in config['users']:
        source_path = os.path.join(base_path, source_category, user)
        output_folder = os.path.join(output_path, config['output'])

        if os.path.exists(source_path):
            # Get all image files
            image_files = []
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']:
                image_files.extend(Path(source_path).glob(ext))
                image_files.extend(Path(source_path).glob(ext.upper()))

            # Also check for files without extension (like IMG_20230724_123456)
            all_files = os.listdir(source_path)
            for f in all_files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                    if Path(os.path.join(source_path, f)) not in image_files:
                        image_files.append(Path(os.path.join(source_path, f)))

            print(f"Found {len(image_files)} images in {source_category}/{user}")

            for img_path in image_files:
                try:
                    # Option 1: Simple zoom and resize
                    # processed_img = zoom_and_resize_image(str(img_path), target_size=(256, 256), zoom_factor=1.3)

                    # Option 2: Advanced - remove background and zoom (recommended)
                    processed_img = remove_background_and_zoom(str(img_path), target_size=(256, 256))

                    if processed_img is not None:
                        # Generate output filename
                        filename = f"{source_category}_{user}_{img_path.stem}.jpg"
                        output_path_full = os.path.join(output_folder, filename)

                        # Save the processed image
                        cv2.imwrite(output_path_full, processed_img)
                        total_processed += 1

                        if total_processed % 10 == 0:
                            print(f"Processed {total_processed} images...")
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
        else:
            print(f"Warning: Path not found: {source_path}")

print(f"\n✅ Processing complete! Total images processed: {total_processed}")
print(f"📁 Output saved to: {output_path}")

# Verify the structure
print("\n📂 Folder structure created:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Show subfolders only, not files
        for d in dirs:
            subindent = ' ' * 2 * (level + 1)
            print(f'{subindent}{d}/')

Found 464 images in Healthy/Ronnel
Processed 10 images...
Processed 20 images...
Processed 30 images...
Processed 40 images...
Processed 50 images...
Processed 60 images...
Processed 70 images...
Processed 80 images...
Processed 90 images...
Processed 100 images...
Processed 110 images...
Processed 120 images...
Processed 130 images...
Processed 140 images...
Processed 150 images...
Processed 160 images...
Processed 170 images...
Processed 180 images...
Processed 190 images...
Processed 200 images...
Processed 210 images...
Processed 220 images...
Processed 230 images...
Processed 240 images...
Processed 250 images...
Processed 260 images...
Processed 270 images...
Processed 280 images...
Processed 290 images...
Processed 300 images...
Processed 310 images...
Processed 320 images...
Processed 330 images...
Processed 340 images...
Processed 350 images...
Processed 360 images...
Processed 370 images...
Processed 380 images...
Processed 390 images...
Processed 400 images...
Processed 410 

## 64x64 Version

### Classification Loaded

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/CornSeeds/NewCornData/processed_data'))

['Healthy_Processed', 'Unhealthy_Processed']


In [ ]:
training_datagenerator1 = ImageDataGenerator(rescale = 1./255)

In [ ]:
testing_set1 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(64, 64),
    batch_size=32,
    class_mode='binary'
)

Found 1985 images belonging to 2 classes.


In [ ]:
cnn_process1 = tf.keras.models.Sequential()

### FIRST LAYER

In [ ]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=[64, 64, 3]
    )
)

cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### SECOND LAYER

In [ ]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
    )
)
cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

 ### PROCESSING

In [ ]:
cnn_process1.add(tf.keras.layers.Flatten())

In [ ]:
cnn_process1.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn_process1.add(tf.keras.layers.Dropout(0.5))

In [ ]:
cnn_process1.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [ ]:
cnn_process1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#STEP 1
#INPUT_SHAPE = 64
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=50)

Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 479s 8s/step - accuracy: 0.6327 - loss: 0.6280 - val_accuracy: 0.7652 - val_loss: 0.5151
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 223ms/step - accuracy: 0.7275 - loss: 0.5066 - val_accuracy: 0.7652 - val_loss: 0.4462
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 227ms/step - accuracy: 0.7526 - loss: 0.4634 - val_accuracy: 0.7708 - val_loss: 0.4227
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7647 - loss: 0.4428 - val_accuracy: 0.7708 - val_loss: 0.4220
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 230ms/step - accuracy: 0.7678 - loss: 0.4196 - val_accuracy: 0.7723 - val_loss: 0.3945
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 227ms/step - accuracy: 0.7592 - loss: 0.4170 - val_accuracy: 0.7718 - val_loss: 0.3960
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7673 - loss: 0.3998 - val_accuracy: 0.7723 - val_loss: 0.3819
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 226ms/step - accuracy: 0.7673 - loss: 0.3906 - val_accura

In [ ]:
cnn_process1.save('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_2-stream.keras')

 # MODEL TESTING (64x64)

In [5]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_process1)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_2-stream.tflite', 'wb') as f:
    f.write(tflite_model)

NameError: name 'cnn_process1' is not defined

In [ ]:
import tensorflow as tf

# Path to your saved model
model_path = '/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_v0_0_1.keras'

# Load the model
print(f"Loading model from {model_path}...")
loaded_cnn_model = tf.keras.models.load_model(model_path)
print("Model loaded successfully!")

In [ ]:
import numpy as np
from keras.preprocessing import image
import os

# Directory for testing images
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Setup generator for testing
prediction_datagen = image.ImageDataGenerator(rescale=1./255)
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

# Run prediction using the LOADED model
predictions = loaded_cnn_model.predict(prediction_generator)

# Mapping and output
idx_to_class = {0: 'healthy', 1: 'unhealthy'}
filenames = prediction_generator.filenames

print("\n--- Predictions using loaded model ---")
for i, score in enumerate(predictions):
    predicted_class = idx_to_class[1 if score[0] > 0.5 else 0]
    image_name = os.path.basename(filenames[i])
    print(f"Image: {image_name}, Predicted: {predicted_class} (Confidence: {score[0]:.4f})")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(64, 64),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

# Counters for statistics
healthy_count = 0
unhealthy_count = 0
processed_count = 0

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        # Update counters
        if predicted_class_idx == 0:
            healthy_count += 1
        else:
            unhealthy_count += 1
        processed_count += 1

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

# Display summary statistics
if processed_count > 0:
    healthy_percent = (healthy_count / processed_count) * 100
    unhealthy_percent = (unhealthy_count / processed_count) * 100

    print("\n" + "="*30)
    print("PREDICTION SUMMARY")
    print("="*30)
    print(f"Total seeds processed: {processed_count}")
    print(f"Healthy seeds:   {healthy_count} ({healthy_percent:.2f}%)")
    print(f"Unhealthy seeds: {unhealthy_count} ({unhealthy_percent:.2f}%)")
    print("="*30)
else:
    print("\nNo images were successfully processed.")

## 128x128 **Version**

### Classification Loaded

In [6]:
import os
print(os.listdir('/content/drive/MyDrive/CornSeeds/NewCornData/processed_data'))

['Healthy_Processed', 'Unhealthy_Processed']


In [7]:
training_datagenerator1 = ImageDataGenerator(rescale = 1./255)

In [8]:
testing_set1 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 1985 images belonging to 2 classes.


In [9]:
cnn_process1 = tf.keras.models.Sequential()

### FIRST LAYER

In [10]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
        input_shape=[128, 128, 3]
    )
)

cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### SECOND LAYER

In [11]:
cnn_process1.add(
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=3,
        activation='relu',
    )
)
cnn_process1.add(
    tf.keras.layers.MaxPool2D(
        pool_size = 2,
        strides = 2
    )
)

### Processing (64x64)

In [12]:
cnn_process1.add(tf.keras.layers.Flatten())

In [13]:
cnn_process1.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn_process1.add(tf.keras.layers.Dropout(0.5))

In [14]:
cnn_process1.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [15]:
cnn_process1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
training_set1 = training_datagenerator1.flow_from_directory(
    '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)
cnn_process1.fit(x=training_set1, validation_data=testing_set1, epochs=50)

Found 1985 images belonging to 2 classes.
Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 635s 10s/step - accuracy: 0.6952 - loss: 0.6140 - val_accuracy: 0.7254 - val_loss: 0.4890
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 164ms/step - accuracy: 0.7723 - loss: 0.4106 - val_accuracy: 0.8670 - val_loss: 0.3645
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 161ms/step - accuracy: 0.8181 - loss: 0.3582 - val_accuracy: 0.7975 - val_loss: 0.3186
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 154ms/step - accuracy: 0.8348 - loss: 0.3378 - val_accuracy: 0.8851 - val_loss: 0.2784
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 145ms/step - accuracy: 0.8751 - loss: 0.2801 - val_accuracy: 0.8987 - val_loss: 0.2492
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 158ms/step - accuracy: 0.8574 - loss: 0.3107 - val_accuracy: 0.8972 - val_loss: 0.2406
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 158ms/step - accuracy: 0.8897 - loss: 0.2495 - val_accuracy: 0.9118 - val_loss: 0.2099
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 148ms/step - acc

In [17]:
cnn_process1.save('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_128x128_v0_0_1-stream.keras')

In [18]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_process1)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open('/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_128x128_v0_0_1-stream.tflite', 'wb') as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpgti9y8cd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  135115895158160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115895159312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115895158736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115895160080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115893408400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115895160656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115893409936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135115893410704: TensorSpec(shape=(), dtype=tf.resource, name=None)


## MODEL TESTING (128x128)

In [19]:
import tensorflow as tf

# Path to your saved model
model_path = '/content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_128x128_v0_0_1-stream.keras'

# Load the model
print(f"Loading model from {model_path}...")
loaded_cnn_model = tf.keras.models.load_model(model_path)
print("Model loaded successfully!")

Loading model from /content/drive/MyDrive/CornSeeds/OUTPUT_CapstoneProject_corn_seed_quality_detection_128x128_v0_0_1-stream.keras...
Model loaded successfully!


In [ ]:
import numpy as np
from keras.preprocessing import image
import os

# Directory for testing images
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Setup generator for testing
prediction_datagen = image.ImageDataGenerator(rescale=1./255)
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

# Run prediction using the LOADED model
predictions = loaded_cnn_model.predict(prediction_generator)

# Mapping and output
idx_to_class = {0: 'healthy', 1: 'unhealthy'}
filenames = prediction_generator.filenames

print("\n--- Predictions using loaded model ---")
for i, score in enumerate(predictions):
    predicted_class = idx_to_class[1 if score[0] > 0.5 else 0]
    image_name = os.path.basename(filenames[i])
    print(f"Image: {image_name}, Predicted: {predicted_class} (Confidence: {score[0]:.4f})")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/corn_data/labeled_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(128, 128),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

Found 3007 images belonging to 3 classes.

--- Starting predictions for 3007 files ---
Image: aug_3_2036712710_WangDataa35.jpg, Predicted: unhealthy (Confidence: 1.0000)
Image: aug_3_203714993_WangDataa46.jpg, Predicted: healthy (Confidence: 0.0289)
Image: aug_3_203818312_WangDataa60.jpg, Predicted: healthy (Confidence: 0.0606)
Image: aug_3_2040426164_WangDataa43.jpg, Predicted: healthy (Confidence: 0.0010)
Image: aug_3_2040816385_WangDataa36.jpg, Predicted: unhealthy (Confidence: 1.0000)
Image: aug_3_2042297005_WangDataa41.jpg, Predicted: healthy (Confidence: 0.0578)
Skipping file healthy/aug_3_2043542508_WangDataa64.jpg due to error: cannot identify image file <_io.BytesIO object at 0x7ae304106610>
Image: healthy_1.jpg, Predicted: unhealthy (Confidence: 1.0000)
Image: healthy_10.jpg, Predicted: healthy (Confidence: 0.0006)
Image: healthy_100.jpg, Predicted: healthy (Confidence: 0.0152)
Image: healthy_1000.jpg, Predicted: healthy (Confidence: 0.0000)
Image: healthy_101.jpg, Predicted:

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os
from pathlib import Path
from PIL import Image

# Path to the directory containing images to be tested
test_data_path = '/content/drive/MyDrive/CornSeeds/NewCornData/processed_data/'

# Create generator
prediction_datagen = image.ImageDataGenerator(rescale=1./255)

# Note: flow_from_directory will still attempt to index all files.
# We will use a safe prediction loop to handle potential corrupted files.
prediction_generator = prediction_datagen.flow_from_directory(
    test_data_path,
    target_size=(128, 128),
    batch_size=1, # Set to 1 for easier individual error handling
    class_mode=None,
    shuffle=False
)

filenames = prediction_generator.filenames
idx_to_class = {0: 'healthy', 1: 'unhealthy'}

# Counters for statistics
healthy_count = 0
unhealthy_count = 0
processed_count = 0

print(f"\n--- Starting predictions for {len(filenames)} files ---")

# Iterate manually to catch UnidentifiedImageError on specific files
for i in range(len(filenames)):
    try:
        # Get the next batch (one image)
        img_batch = next(prediction_generator)

        # Predict
        prediction_score = cnn_process1.predict(img_batch, verbose=0)

        # Process result
        predicted_class_idx = 1 if prediction_score[0][0] > 0.5 else 0
        predicted_class_name = idx_to_class[predicted_class_idx]
        image_name = os.path.basename(filenames[i])

        # Update counters
        if predicted_class_idx == 0:
            healthy_count += 1
        else:
            unhealthy_count += 1
        processed_count += 1

        print(f"Image: {image_name}, Predicted: {predicted_class_name} (Confidence: {prediction_score[0][0]:.4f})")

    except Exception as e:
        print(f"Skipping file {filenames[i]} due to error: {e}")
        continue

# Display summary statistics
if processed_count > 0:
    healthy_percent = (healthy_count / processed_count) * 100
    unhealthy_percent = (unhealthy_count / processed_count) * 100

    print("\n" + "="*30)
    print("PREDICTION SUMMARY")
    print("="*30)
    print(f"Total seeds processed: {processed_count}")
    print(f"Healthy seeds:   {healthy_count} ({healthy_percent:.2f}%)")
    print(f"Unhealthy seeds: {unhealthy_count} ({unhealthy_percent:.2f}%)")
    print("="*30)
else:
    print("\nNo images were successfully processed.")

Found 1985 images belonging to 2 classes.

--- Starting predictions for 1985 files ---
Image: Healthy_Marjorie_IMG_7522.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7523.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7524.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7525.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7526.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7527.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7528.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7529.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7530.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7531.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7532.jpg, Predicted: healthy (Confidence: 0.0000)
Image: Healthy_Marjorie_IMG_7533.jpg, Predicted: health